In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

In [ ]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

In [ ]:

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

df = df.merge(
    flood_rows[['datetime', 'event_id']],
    on='datetime',
    how='left'
)

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


In [ ]:
LOOKAHEAD_HOURS = 24  # predict within next 24h
FREQ_MINUTES = 15     # Hydraulic data's sampling interval 

# Cqalculate how many rows ahead
lookahead_steps = int(LOOKAHEAD_HOURS * 60 / FREQ_MINUTES)

# 'will_flood' checks whether the flood stage will be exceeded in the next `lookahead_steps` readings, setting it as 
# 1 if any of the next readings exceed the flood action stage, and 0 otherwise. This is used as the target variable for the model.

df['will_flood'] = (
    df['gage_height_ft']
    .shift(-1)                                   
    .rolling(window=lookahead_steps, min_periods=1)
    .max()
    .shift(-(lookahead_steps - 1))                # align window to start right after current row
    > FLOOD_ACTION_STAGE
).astype(int)



In [ ]:
import numpy as np

# make a column called gage_height_roc_1h and gage_height_roc_6h to see the rate of change in gage height over 1 hour and 6 hours, respectively 
df['gage_height_roc_1h'] = df['gage_height_ft'].diff(int(60/FREQ_MINUTES))
df['gage_height_roc_6h'] = df['gage_height_ft'].diff(int(360/FREQ_MINUTES))

df['gage_height_now'] = df['gage_height_ft']
df['streamflow_now'] = df['streamflow_cfs']

for col in ['precip_3hr', 'precip_24hr', 'precip_72hr', 'turbidity_fnu']:
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

feature_columns = [
    # hydraulic — current state + trend
    'gage_height_ft',
    'gage_height_roc_1h',
    'gage_height_roc_6h',

    # precip — log-transformed versions only (not the raw skewed ones)
    'precip_3hr_log',
    'precip_24hr_log',
    'precip_72hr_log',

    # weather
    'temperature_2m',
    'wind_speed_10m',
    'vapour_pressure_deficit',
    'rain',
    'snowfall',
    'snow_depth',

    # water quality
    'specific_conductance_us_cm',
    'temperature_c',
]


In [ ]:
# tag every row (not just flood rows) with which event's "influence window" it falls in
# so the same storm doesn't appear in both train and test
events_sorted = events.sort_values('start').reset_index(drop=True)

# hold out the most recent ~20% of events as test
n_test_events = int(len(events_sorted) * 0.2)
test_events = events_sorted.iloc[-n_test_events:]
train_events = events_sorted.iloc[:-n_test_events]

test_start_cutoff = test_events['start'].min() - pd.Timedelta(days=3)  # buffer before first test storm

train_df = df[df['datetime'] < test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])
test_df  = df[df['datetime'] >= test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])

test_df

In [ ]:
"""This will be where the XGBoost model is going to be created."""